In [1]:
!date

Sun Sep  6 13:43:17 PDT 2026


In [2]:
import pandas as pd
import numpy as np
import glob
import os
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

projdir = '/u/project/cluo/terencew/claude/project_ideas/pool_design'
sample = '20220928-IGVF-D0'

In [3]:
### discover pools with at least one modality demuxlet'd so far
pools = sorted(set(p.split('/ambisim/')[1].split('/')[0] for p in
    glob.glob(f'{projdir}/ambisim/*/demux/demuxlet/*/{sample}.best')))
len(pools)

33

In [4]:
### join demuxlet .best calls against ambisim's known donor-of-origin / ambient RNA ground truth
def load_pool_modality(pool, modality):
    best_path = f'{projdir}/ambisim/{pool}/demux/demuxlet/{modality}/{sample}.best'
    if not os.path.exists(best_path):
        return None
    best = pd.read_csv(best_path, sep='\t')
    best['barcode'] = best['BARCODE'].str.replace('-1', '', regex=False)
    truth = pd.read_csv(f'{projdir}/ambisim/{pool}/{sample}/drop_data_rand.txt', sep='\t',
                         dtype={'sam': str, 'ct': str})
    # both GEX and ATAC demuxlet were run against the same RNA-space barcode list
    # (A02b_demuxlet_call.sh passes one $BARCODES to both calls), so ATAC .best
    # BARCODE is already translated into RNA_BC space, not raw ATAC_BC
    truth = truth.rename(columns={'RNA_BC': 'barcode'})
    df = truth.merge(best[['barcode', 'DROPLET.TYPE', 'SNG.BEST.GUESS', 'DIFF.LLK.BEST.NEXT']],
                      on='barcode', how='inner')
    if modality == 'gex':
        df['ambient_frac'] = df['rna_nr_a'] / (df['rna_nr_a'] + df['rna_nr_c'])
    else:
        df['ambient_frac'] = df['atac_nr_a'] / (df['atac_nr_a'] + df['atac_nr_c'])
    df['called_donor'] = df['SNG.BEST.GUESS'].str.split(',').str[0]
    df['is_true_singlet'] = df['n'] == 1
    df['called_singlet'] = df['DROPLET.TYPE'] == 'SNG'
    df['correct'] = df['is_true_singlet'] & df['called_singlet'] & (df['called_donor'] == df['sam'])
    df['ll_gap'] = df['DIFF.LLK.BEST.NEXT']
    universe, strategy, rep = pool.split('__')
    df['pool'] = pool
    df['universe'] = universe
    df['strategy'] = strategy
    df['rep'] = int(rep.replace('rep', ''))
    df['modality'] = modality
    df = df.rename(columns={'sam': 'true_donor'})
    return df[['pool', 'universe', 'strategy', 'rep', 'modality', 'barcode',
               'is_true_singlet', 'called_singlet', 'correct', 'ambient_frac', 'll_gap',
               'called_donor', 'true_donor']]

def _load_job(job):
    return load_pool_modality(*job)

In [5]:
### load every pool x modality in parallel
jobs = [(pool, modality) for pool in pools for modality in ['gex', 'atac']]
with ProcessPoolExecutor(max_workers=10) as ex:
    results = list(tqdm(ex.map(_load_job, jobs), total=len(jobs)))
scored = pd.concat([r for r in results if r is not None], ignore_index=True)

  0%|          | 0/66 [00:00<?, ?it/s]

  2%|▏         | 1/66 [00:01<01:58,  1.82s/it]

  6%|▌         | 4/66 [00:01<00:23,  2.61it/s]

 17%|█▋        | 11/66 [00:03<00:14,  3.71it/s]

 21%|██        | 14/66 [00:03<00:10,  4.95it/s]

...

 85%|████████▍ | 56/66 [00:10<00:01,  5.48it/s]

 92%|█████████▏| 61/66 [00:10<00:00,  8.49it/s]

 97%|█████████▋| 64/66 [00:11<00:00,  5.47it/s]

100%|██████████| 66/66 [00:11<00:00,  5.95it/s]

100%|██████████| 66/66 [00:11<00:00,  5.54it/s]

In [6]:
scored.head()

,pool,universe,strategy,rep,modality,barcode,is_true_singlet,called_singlet,correct,ambient_frac,ll_gap,called_donor,true_donor
0,AFR_only__greedy_maxmin__rep1,AFR_only,greedy_maxmin,1,gex,AAACAGCCAAACAACA,True,True,True,0.098320,13.41,NA20334,NA20334
1,AFR_only__greedy_maxmin__rep1,AFR_only,greedy_maxmin,1,gex,AAACAGCCAAACATAG,True,True,True,0.156914,15.59,NA19042,NA19042
2,AFR_only__greedy_maxmin__rep1,AFR_only,greedy_maxmin,1,gex,AAACAGCCAAACCCTA,True,True,True,0.391450,0.05,HG03081,HG03081
3,AFR_only__greedy_maxmin__rep1,AFR_only,greedy_maxmin,1,gex,AAACAGCCAAACCTAT,True,False,False,0.314493,0.00,HG02678,HG02678
4,AFR_only__greedy_maxmin__rep1,AFR_only,greedy_maxmin,1,gex,AAACAGCCAAACCTTG,True,True,True,0.184638,30.66,NA19707,NA19707


In [7]:
scored.shape

(584935, 13)

In [8]:
### only keep pools where both gex and atac demuxlet finished (some are still mid-run)
complete_pools = scored.groupby('pool')['modality'].nunique()
complete_pools = complete_pools[complete_pools == 2].index
print(len(complete_pools), 'of', len(pools), 'pools have both modalities scored')
scored = scored[scored['pool'].isin(complete_pools)].reset_index(drop=True)

32 of 33 pools have both modalities scored


In [9]:
scored.shape

(575936, 13)

In [10]:
### write checkpoint for downstream pool-level / droplet-level notebooks
outdir = f'{projdir}/csv/ambisim'
os.makedirs(outdir, exist_ok=True)
scored.to_csv(f'{outdir}/droplet_scores.csv', sep='\t', index=False)

In [11]:
!date

Sun Sep  6 13:43:34 PDT 2026
